In [1]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd() / "mutual_fund_project"

nav_path = project_root / "data" / "raw" / "02_nav_history.csv"

print(nav_path)

df = pd.read_csv(nav_path)

print(df.head())

# 1. Parse dates
df['date'] = pd.to_datetime(df['date'])

# 2. Sort by fund code + date
df = df.sort_values(['amfi_code', 'date'])

# 3. Forward-fill missing NAV (holidays = no trading)
df['nav'] = df.groupby('amfi_code')['nav'].ffill()

# 4. Remove duplicates
df = df.drop_duplicates(subset=['amfi_code', 'date'])

# 5. Validate NAV > 0
df = df[df['nav'] > 0]

C:\Users\Shruti Singhania\mutual_fund_project\data\raw\02_nav_history.csv
   amfi_code        date      nav
0     119551  2022-01-03  54.3856
1     119551  2022-01-04  54.3474
2     119551  2022-01-05  54.6869
3     119551  2022-01-06  55.4550
4     119551  2022-01-07  55.3692


In [2]:
from IPython.display import FileLink
df.to_csv("clean_nav.csv", index=False)
FileLink("clean_nav.csv")

C:\Users\Shruti Singhania\clean_nav.csv

In [3]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd() / "mutual_fund_project"

path = project_root / "data" / "raw" / "08_investor_transactions.csv"

print(path)

df1 = pd.read_csv(path)

print(df1.head())

# 1. Standardise transaction_type
df1['transaction_type'] = df1['transaction_type'].str.strip().str.upper()

# 2. Validate amount > 0
df1 = df1[df1['amount_inr'] > 0]

# 3. Check KYC status values
valid_kyc = ['Verified', 'Pending']
df1 = df1[df1['kyc_status'].isin(valid_kyc)]

# 4. Fix date formats
df1['transaction_date'] = pd.to_datetime(df1['transaction_date'], dayfirst=True, format='mixed', errors='coerce')

C:\Users\Shruti Singhania\mutual_fund_project\data\raw\08_investor_transactions.csv
  investor_id transaction_date  amfi_code transaction_type  amount_inr  \
0   INV003054       01-01-2024     119092              SIP        1834   
1   INV002952       01-01-2024     148567       Redemption      392882   
2   INV003420       01-01-2024     118636              SIP         912   
3   INV003436       01-01-2024     118634              SIP        1102   
4   INV004691       01-01-2024     119094          Lumpsum        8682   

         state       city city_tier age_group  gender  annual_income_lakh  \
0    Telangana  Hyderabad       T30       56+  Female                77.1   
1       Punjab   Amritsar       B30     18-25    Male                 7.1   
2      Haryana  Faridabad       B30     36-45    Male                47.2   
3  Maharashtra     Mumbai       T30     36-45  Female                54.4   
4        Delhi      Noida       T30     26-35    Male                14.5   

  paymen

In [4]:
from IPython.display import FileLink
df1.to_csv("clean_investor.csv", index=False)
FileLink("clean_investor.csv")

C:\Users\Shruti Singhania\clean_investor.csv

In [5]:
import numpy as np
from pathlib import Path
import pandas as pd

project_root = Path.cwd() / "mutual_fund_project"

path = project_root / "data" / "raw" / "07_scheme_performance.csv"

print(path)

df2 = pd.read_csv(path)

print(df2.head())

# 1. Validate return values are numeric
return_cols = ['return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct']
for col in return_cols:
    df2[col] = pd.to_numeric(df2[col], errors='coerce')

# 2. Flag negative Sharpe ratios (don't delete — just flag)
df2['negative_sharpe_flag'] = df2['sharpe_ratio'] < 0

# 3. Check expense_ratio range (0.1% – 2.5%)
df2 = df2[(df2['expense_ratio_pct'] >= 0.1) & (df2['expense_ratio_pct'] <= 2.5)]

C:\Users\Shruti Singhania\mutual_fund_project\data\raw\07_scheme_performance.csv
   amfi_code                                   scheme_name       fund_house  \
0     119551     SBI Bluechip Fund - Regular Plan - Growth  SBI Mutual Fund   
1     119552      SBI Bluechip Fund - Direct Plan - Growth  SBI Mutual Fund   
2     119598    SBI Small Cap Fund - Regular Plan - Growth  SBI Mutual Fund   
3     119599     SBI Small Cap Fund - Direct Plan - Growth  SBI Mutual Fund   
4     119120  SBI Magnum Gilt Fund - Regular Plan - Growth  SBI Mutual Fund   

    category     plan  return_1yr_pct  return_3yr_pct  return_5yr_pct  \
0  Large Cap  Regular           12.42           12.36           14.45   
1  Large Cap   Direct           15.25           11.30           14.23   
2  Small Cap  Regular           24.56           23.39           20.67   
3  Small Cap   Direct           20.59           23.14           21.82   
4       Gilt  Regular            5.34            6.07            5.43   

   be

In [6]:
from IPython.display import FileLink
df2.to_csv("clean_performance.csv", index=False)
FileLink("clean_performance.csv")

C:\Users\Shruti Singhania\clean_performance.csv